<a href="https://colab.research.google.com/github/rafahcs/Projeto_AF/blob/main/projetoaf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. INTRODUÇÃO


* O problema consiste em um autômato finito determinístico(AFD) que reconheça a linguagaem formada pelas notações dos elementos pertencentes aos conjuntos dos números inteiros, números reais, números hexadecimais, números em notação científica e números complexos.

* O AFD recebe uma fita de tamanho variável. Cada estado armazena o último símbolo anterior lido após a transição do estado antecesssor.

* Uma palavra só é aceita se o útimo símbolo for lido por um estado que seja final. Palavras não aceitas são rejeitadas e o autômato libera o armazenamento da fita e lê a próxima palavra.

* Se o "valor" de uma expressão regular(ER) é uma linguagem, Então as linguagem dos inteiros($L_{int}$), linguagem dos reais($L_{real}$), linguagem de notação científica($L_{cient}$), linguagem dos hexadecimais($L_{hex}$) e linguagem dos complexos($L_{comp}$) podem ser escritas como ER's. A união dessas ER's forma a linguagem do AFD A. Dessa forma, a linguagem escolhida é L(A) = $L_{int}$ ∪ $L_{real}$ ∪ $L_{cient}$ ∪ $L_{hex}$ ∪ $L_{comp}$.

  ' | ' → pipe utilizado como operador 'ou' em na expressão regular "(+ou-)", pois '+' é um símbolo nas linguagens

  $L_{int}$ = (+ | -)Σ$^{*}$ , onde Σ = {0,1,2,3,4,5,6,7,8,9}
  
  $L_{real}$ = (+ | -)Σ$^{*}$ . Σ +

  $L_{cient}$ = $L_{real}(e + E)(+ | -)Σ$^{*}$ +

  $L_{hex}$ = $[0(x + X)Σ +] + [$\#$Σ +] + [Σ +(h + H)]$, onde Σ = [0,9] ∪ [A,F] ∪ [a, f]

  $L_{comp}$ = $[L_{real}(+ | -)L_{real} i ] + (L_{real})$

  L(A) = { w | w é uma palavra real ou inteira ou complexa ou notação científica ou hexadecimal}

* A estratégia adotada foi a de execução da fita por cada autômato representado por uma função builder_ que percorre a fita lendo cada símbolo e comparando com o valor esperado para o alfabeto do autômato. Ao final são rodados os testes unitários que retornam se a fita tem uma palavra aceita ou não na linguagem.


# 2. DEFINIÇÃO FORMAL DO AUTÔMATO FINITO


A = (Q, Σ, δ, qO, F)

Q: Um conjunto finito de estados.

Σ: Um alfabeto finito de símbolos (alfabeto da fita).

δ: Função de transição

qO: Estado inicial

F: Conjunto de estados finais ou de aceitação

* Q = $Q_{int}$ ∪ $Q_{real}$ ∪ $Q_{cient}$ ∪ $Q_{hex}$ ∪ $Q_{comp}$

    = {q0, q1, q2} ∪ {q0, q1, q2, q3, q4, q5} ∪ {q0, q1, q2, q3, q4, q5, q6, q7} ∪ {q0, q1, q2, q3, q4, q5} ∪ {q0, q1, q2, q3, q4, q5, q6}

    = {q0, q1, q2, q3, q4, q5, q6, q7}

* Σ = {+, -, .} ∪ $\mathbb{Z}$ ∪ $\mathbb{R}$ ∪ {A-F,H,a-f,h} ∪ $\mathbb{C}$

* δ = Q x Σ $\rightarrow$ Q

* q0 = {q0}

* F = $F_{int}$ ∪ $F_{real}$ ∪ $F_{cient}$ ∪ $F_{hex}$ ∪ $F_{comp}$

    = {q2} ∪ {q2, q4, q5} ∪ {q7} ∪ {q4, q5} ∪ {q6}
    
    = {q2, q4, q5, q6, q7}

5-upla completa:

A = {
      {q0, q1, q2, q3, q4, q5, q6, q7},
      {+, -, .} ∪ $\mathbb{Z}$ ∪ $\mathbb{R}$ ∪ {A,B,C,D,E,F,H,a,b,c,d,e,f,h} ∪ $\mathbb{C}$,
      Q x Σ $\rightarrow$ Q,
      {q0},
      {q2, q4, q5, q6, q7}
    }

# 3. IMPLEMENTAÇÃO


In [6]:
from typing import Dict, Tuple


class FiniteAutomata:
    delta: Dict[Tuple[str, str], str] = None

#inicialização do automato
    def __init__(self, tape, initial_state, final_states, delta):
        self.tape = list(tape)  #cria fita
        self.blank = "_"
        self.head = 0 #cabeça de leitura da fita

        self.initial_state = initial_state  #estado inicial
        self.current_state = initial_state  #estado atual
        self.final_states = final_states  #estados finais
        self.delta = delta  #estados de aceitação

#leitura de símbolo
    def read_symbol(self):
        if self.head >= len(self.tape):
            return self.blank
        return self.tape[self.head]

#função de transição
    def step(self):
        symbol = self.read_symbol()
        key = (self.current_state, symbol)

        if key not in self.delta:
            return False

        self.current_state = self.delta[key]
        self.head += 1
        return True

    def execute(self):
        while self.head < len(self.tape):
            if not self.step():
                return False
        return self.current_state in self.final_states


# INTEIRO
def build_integer_af(tape):
    delta = {}
    digits = "0123456789"

    delta[("q0", "+")] = "q1"
    delta[("q0", "-")] = "q1"

    for d in digits:
        delta[("q0", d)] = "q2"
        delta[("q1", d)] = "q2"
        delta[("q2", d)] = "q2"

    return FiniteAutomata(tape, "q0", {"q2"}, delta)


# REAL
def build_real_af(tape):
    delta = {}
    digits = "0123456789"

    # início
    delta[("q0", "+")] = "q1"
    delta[("q0", "-")] = "q1"
    delta[("q0", ".")] = "q3"

    # número inteiro inicial
    for d in digits:
        delta[("q0", d)] = "q2"
        delta[("q1", d)] = "q2"
        delta[("q2", d)] = "q2"

    # ponto depois de número → aceita "3."
    delta[("q2", ".")] = "q5"

    # ponto sem número antes → precisa de dígito depois
    for d in digits:
        delta[("q3", d)] = "q4"

    # parte decimal normal
    for d in digits:
        delta[("q4", d)] = "q4"
        delta[("q5", d)] = "q4"

    # sinal seguido de ponto
    delta[("q1", ".")] = "q3"

    return FiniteAutomata(
        tape,
        "q0",
        {"q2", "q4", "q5"},  # <-- chave da correção
        delta
    )


# CIENTÍFICA
def build_scientific_af(tape):
    delta = {}
    digits = "0123456789"

    delta[("q0", "+")] = "q1"
    delta[("q0", "-")] = "q1"
    delta[("q0", ".")] = "q3"

    for d in digits:
        delta[("q0", d)] = "q2"
        delta[("q1", d)] = "q2"
        delta[("q2", d)] = "q2"
        delta[("q3", d)] = "q4"
        delta[("q4", d)] = "q4"

    delta[("q1", ".")] = "q3"
    delta[("q2", ".")] = "q3"
    delta[("q3", "e")] = "q5"
    delta[("q3", "E")] = "q5"
    delta[("q2", "e")] = "q5"
    delta[("q2", "E")] = "q5"
    delta[("q4", "e")] = "q5"
    delta[("q4", "E")] = "q5"

    delta[("q5", "+")] = "q6"
    delta[("q5", "-")] = "q6"

    for d in digits:
        delta[("q5", d)] = "q7"
        delta[("q6", d)] = "q7"
        delta[("q7", d)] = "q7"

    return FiniteAutomata(tape, "q0", {"q7"}, delta)


# HEXADECIMAL
def build_hexadecimal_af(tape):
    delta = {}
    hex_digits = "0123456789abcdefABCDEF"

    # 0x...
    delta[("q0", "0")] = "q1"
    delta[("q1", "x")] = "q2"
    delta[("q1", "X")] = "q2"

    # #ABC
    delta[("q0", "#")] = "q3"

    for d in hex_digits:
        if d != "0": # Se for 0, já definimos que vai para q1
            delta[("q0", d)] = "q4"
        delta[("q1", d)] = "q4" # Começando direto (ex: 1A3)
        delta[("q2", d)] = "q4"
        delta[("q3", d)] = "q4"
        delta[("q4", d)] = "q4"

    delta[("q4", "h")] = "q5"
    delta[("q4", "H")] = "q5"

    return FiniteAutomata(tape, "q0", {"q4", "q5"}, delta)


# COMPLEXO
def build_complex_af(tape):
    delta = {}
    digits = "0123456789"

    # --- INÍCIO ---
    delta[("q0", "+")] = "q1"
    delta[("q0", "-")] = "q1"
    delta[("q0", "i")] = "q6"

    # número inicial (parte real OU imaginária direta)
    for d in digits:
        delta[("q0", d)] = "q2"
        delta[("q1", d)] = "q2"
        delta[("q2", d)] = "q2"

    # ponto decimal (opcional)
    delta[("q2", ".")] = "q3"
    for d in digits:
        delta[("q3", d)] = "q3"

    # número seguido de i → forma "bi"
    delta[("q1", "i")] = "q6"
    delta[("q2", "i")] = "q6"
    delta[("q3", "i")] = "q6"

    # parte real seguida de + ou -
    delta[("q2", "+")] = "q4"
    delta[("q2", "-")] = "q4"
    delta[("q3", "+")] = "q4"
    delta[("q3", "-")] = "q4"

    # após sinal da parte imaginária:
    delta[("q4", "i")] = "q6"  # aceita "3+i"

    for d in digits:
        delta[("q4", d)] = "q5"

    # parte imaginária numérica
    for d in digits:
        delta[("q5", d)] = "q5"

    delta[("q5", ".")] = "q4"
    delta[("q5", "i")] = "q6"

    return FiniteAutomata( tape, "q0", {"q6"}, delta )


# construtor dos automatos
automata = {
        "Inteiro": build_integer_af,
        "Real": build_real_af,
        "Cientifico": build_scientific_af,
        "Hexadecimal": build_hexadecimal_af,
        "Complexo": build_complex_af
    }


# TESTES
def run_tests():
    tests = {
        "Inteiro": [
            ("1", True), ("+2", True), ("-3", True),                                # Pequenas/Óbvias
            (" ", False), ("+", False), ("-", False), ("+-12", False),              # Malformadas
            ("13c", False), ("++10", False), ("+12bc", False)                       # Inválidas
        ],
        "Real": [
            ("3.14", True), ("-0.5", True), ("+2.0", True), (".5", True), ("5.", True), # Óbvias
            ("1.2.3", False), ("+-2.5", False), (".", False),                           # Malformadas
            ("e.5", False), ("1,5", False)                                              # Inválidas
        ],
        "Cientifico": [
            ("1e0", True), ("6E+7", True), ("-5e-10", True),  # Óbvias
            ("e10", False), ("1e", False), ("1e+-2", False),  # Malformadas
            ("1.5e3.2", False), ("2 e 2", False)              # Inválidas
        ],
        "Hexadecimal": [
            ("0xxF", True), ("0xF", True), ("#1", True), ("Fh", True),        # Óbvias
            ("0x", False), ("#", False), ("h", False),        # Malformadas
            ("0xG1", False), ("#ZZ", False), ("1A3Xh", False) # Inválidas

        ],
        "Complexo": [
            ("i", True), ("+i", True), ("-i", True),               # Pequenas
            ("3+4i", True), ("2.5-1.3i", True),                    # Óbvias
            ("2--i", False), ("i+3", False), ("6+4j", False),      # Malformadas
            ("2.5.3i", False), ("2.5.3+i", False), ("+-i", False)  # Inválidas
        ]
    }

    for name, builder in automata.items():
        print(f"\n=== {name} ===")
        for t, expected in tests.get(name, []):
            result = builder(t).execute()

            if result == expected:
                print(f"OK   {t}")
            else:
                print(f"FAIL {t} esperado {expected}, obteve {result}")


# EXECUÇÃO
if __name__ == "__main__":
    run_tests()


=== Inteiro ===
OK   1
OK   +2
OK   -3
OK    
OK   +
OK   -
OK   +-12
OK   13c
OK   ++10
OK   +12bc

=== Real ===
OK   3.14
OK   -0.5
OK   +2.0
OK   .5
OK   5.
OK   1.2.3
OK   +-2.5
OK   .
OK   e.5
OK   1,5

=== Cientifico ===
OK   1e0
OK   6E+7
OK   -5e-10
OK   e10
OK   1e
OK   1e+-2
OK   1.5e3.2
OK   2 e 2

=== Hexadecimal ===
FAIL 0xxF esperado True, obteve False
OK   0xF
OK   #1
OK   Fh
OK   0x
OK   #
OK   h
OK   0xG1
OK   #ZZ
OK   1A3Xh

=== Complexo ===
OK   i
OK   +i
OK   -i
OK   3+4i
OK   2.5-1.3i
OK   2--i
OK   i+3
OK   6+4j
OK   2.5.3i
OK   2.5.3+i
OK   +-i


# 4. TESTES

    1. Instâncias pequenas;
    2. Instâncias inválidas malformadas;
    3. Casos óbvios de aceitação/rejeição.


In [5]:
# TESTES
def run_tests():
    tests = {
        "Inteiro": [
            ("1", True), ("+2", True), ("-3", True),                                # Pequenas/Óbvias
            (" ", False), ("+", False), ("-", False), ("+-12", False),              # Malformadas
            ("13c", False), ("++10", False), ("+12bc", False),                      # Inválidas
        ],
        "Real": [
            ("3.14", True), ("-0.5", True), ("+2.0", True), (".5", True), ("5.", True), # Óbvias
            ("1.2.3", False), ("+-2.5", False), (".", False),                           # Malformadas
            ("e.5", False), ("1,5", False)                                              # Inválidas
        ],
        "Cientifico": [
            ("1e0", True), ("6E+7", True), ("-5e-10", True),  # Óbvias
            ("e10", False), ("1e", False), ("1e+-2", False),   # Malformadas
            ("1.5e3.2", False), ("2 e 2", False),             # Inválidas
        ],
        "Hexadecimal": [
            ("0xF", True), ("#1", True), ("Fh", True),        # Óbvias
            ("0x", False), ("#", False), ("h", False),        # Malformadas
            ("0xG1", False), ("#ZZ", False), ("1A3Xh", False) # Inválidas

        ],
        "Complexo": [
            ("i", True), ("+i", True), ("-i", True),               # Pequenas
            ("3+4i", True), ("2.5-1.3i", True),                    # Óbvias
            ("2--i", False), ("i+3", False), ("6+4j", False),      # Malformadas
            ("2.5.3i", False), ("2.5.3+i", False), ("+-i", False)  # Inválidas
        ]
    }

    for name, builder in automata.items():
        print(f"\n=== {name} ===")
        for t, expected in tests.get(name, []):
            result = builder(t).execute()

            if result == expected:
                print(f"OK   {t}")
            else:
                print(f"FAIL {t} esperado {expected}, obteve {result}")

# 5. DEMONSTRAÇÃO FORMAL

Seja A um autômato finito e L uma linguagem. Dizemos que A é correto em relação à L, se L = L(A), ou seja, se A reconhece L. Quando L está clara pelo contexto, podemos dizer apenas que A é correto.

Para provar a corretude de um AFD A = (Q, Σ, δ, q0, F), é fundamental que seja estabelecido, quando necessário, que propriedades uma cadeia x deve ter para que $\hat{δ}$(q0, x) = q para cada q ∈ Q.

Porém, A é formado por autômatos distintos. Então, para a provar a corretude de A é preciso fazer uma análise isolada da corretude de cada autômato pertencente a A.

## 5.1 Prova de corretude do AF inteiros

  $L_{int}$ = { sw | s ∈ {+,-} $\land$ w ∈ {0,1,2,..,9} } sob o alfabeto Σ = {+,-,0,1,2,..,9}

  Primeiramente, vamos provar as seguinte propriedades usando indução estrutural:

  (1) δ(q0, x) = q2 ⇒ x ∈ {0,1,2,..,9}

  (2) δ(q0, x) = q1 ⇒ x = '+' $\lor$ x = '-'

  **Caso base (x é uma palavra de tamanho 1)**

  Como q2 ∈ F, a palavra é aceita. Com isso, a premissa (1) está provada.

  Como q1 ∉ F, sendo F = {q2}, a palavra é rejeitada. Logo, como a premissa é falsa, implicação (2) está provada.

  **Prova por Indução**

  Seja x = wa, onde w é uma cadeia no alfabeto Σ e a ∈ Σ é um símboo do alfabeto. Supondo que as implicações (1) e (2) são válidas para w, é preciso provar que também são válidas para x.
  
  **Hipóteses de indução:**

  Assumimos que para uma paavra x de comprimento $n$, o autômato está em:
  - q2 se x for um inteiro válido.
  - q1 se x for apenas um sinal.

  **Passo Indutivo (Palavra w + s de comprimento $n+1$)**
  1. Se x ∈ q2 e s ∈ d, onde d = {0, ... ,9}, então $\hat{δ}$(q2, s) = q2. O autômato permanece em estado de aceitação. Isso permite números de qualquer tamanho (ex: "123").

  2. Se x ∈ q1 e s ∈ d: δ(q1, s) = q2. O autômato passa a aceitar (ex: "+1").

  3. Transições Inválidas: Se x ∈ q2 e s ∈ {+, -}, não há transição definida em $\delta$. O método step() retornará False, resultando em rejeição. Isso impede casos como "12+3".

  **Conclusão:** Por indução, o AFD aceita precisamente a sequência: $[Sinal] Dígito$.

## 5.2 Prova de corretude do AF real

  -  q3 e q4 tratam a obrigatoriedade de dígitos após o ponto se a string começar com ponto (ex: .5).

  - q5 permite o formato "Número seguido de ponto" (ex: 5.), comum em linguagens de programação.

  - Corretude: O conjunto F = {q2, q4, q5} cobre todas as formas válidas de números reais, enquanto a ausência de transições de q3 para F sem um dígito garante que apenas ' **.** ' seja rejeitado.

## 5.3 Prova de corretude do AF hexadecimal

  Este autômato valida três formatos distintos:

  a. Prefixo 0x:

    δ(q0, 0) = q1
    δ(q0, x) = q1
    δ(q0, hex) = q1, onde hex é um algarismo do alfabeto hexadecimal

  b. Prefixo #:

    δ(q0, #) = q3
    δ(q3, hex) = q4
  
  c. Sufixo h:

    δ(q4, h) = q5

  Prova de Rejeição: Se uma string for 0xG, o símbolo G não existe no dicionário para o estado $q_2$, forçando a falha imediata.

## 5.4 Prova de corretude do AF notação científica

  Caso base: a menor palavra em notação científica é ded ou dEd, onde $d$ é o dígito coeficiente e $e$ ou $E$ é o expoente da base 10. As transições para aceitação dessa palavra são

    δ(q0, d) = q2
    δ(q2, e) = δ(q2, E) = q5
    δ(q5, d) = q7

  Como q7 ∈ $F$, sendo F o conjunto de estados finais do autômato, o caso base é aceito.

  **Hipótese de indução**

  Assumimos que qualquer palavra $s$ que represente um prefixo válido de um número científico levará o autômato a um estado condizente com sua estrutura (ex: se terminou em $E$, está em q5).

  **Passo indutivo: Extensões e Restrições**

  1. **Coeficiente Real**: Se a palavra possui ponto (ex: 1.5e), o AFD transita

    δ(q2, .) = q3
    δ(q3, d) = q4
    δ(q2, e) = q5
  
   Isso garante que o coeficiente real possa ser decimal.
   
  2. **Sinal do Expoente**: A partir de q5 ($e$), o autômato pode ir para q6 (sinal) ou direto para q7 (dígito). Isso prova a opcionalidade do sinal do expoente.
  
  3. **Fechamento de Dígitos**: Em q7, qualquer dígito adicional mantém o estado em q7 ( δ(q7, d) = q7 ). Isso permite expoentes com múltiplos dígitos (ex: e100).

  **Prova de rejeição (Solidez)**

  O AFD deve rejeitar strings malformadas. Analisando dois casos.

  *  **Caso 1**: Expoente sem dígitos ("1e" ou "1E")
      * Caminho:

            δ(q0, d) = q2
            δ(q2, e) = δ(q2, E) = q5

      * O estado final é q5. Como q5 ∉ F, a palavra é rejeitada. Correto.
  * **Caso 2**: Sinais duplicados ("1e+-2")
      * Caminho:

            δ(q0, d) = q2
            δ(q2, e) = δ(q2, E) = q5
            δ(q5, +) = q6
            δ(q5, -) = q6

      * Em q6, o único símbolo permitido é um dígito. Se o próximo símbolo for -, não há transição definida em δ para (q6, "-"). O método step() retornará False, resultando em rejeição. Correto.

## 5.5 Prova de corretude do AF complexo

  A corretude é garantida pois o estado final q6 só é alcançável através do símbolo $i$. Se a string terminar em dígito (como "3+4"), ela para em q5 e é rejeitada, pois números complexos exigem a unidade imaginária neste autômato específico.

# 6. CONCLUSÃO

## Limitações

- Limitação do modelo AFD, que não possuem memória limitadas, ou seja,  não conseguem lidar com estruturas mais complexas que exigem “lembrar” muitas informações.

- Testes não cobrem 100% dos casos. há um número finito de casos e palavras específicas. Isso limita o AFD a não ser testado em todas as combinações de palavras possíveis.

- Sensibilidade a Espaços e Caracteres Invisíveis: O autômato falha se a fita contiver espaços no início ou no fim (ex: " 123 "), pois ele tenta processar o espaço como um símbolo e não encontra transição na função delta.

## Aprendizados

- Compreensão do funcionamento de um AFD
- Importância de tratar casos de erro e entradas inválidas